# Notebook 01 — Full supervised fine-tuning (SFT)

**Workshop:** Agentic AI for Scientists — Week 4 (Post-Training & Deployment)
**Block:** full fine-tuning (25 min)
**Goal:** Update **100% of the weights** of `Qwen/Qwen3-0.6B` so it learns the clinical-extraction behavior from Notebook 00. This is the gold-standard, highest-quality (and most expensive) form of post-training. Notebooks 02 (LoRA) and 03 (QLoRA) will reach almost the same quality for a fraction of the cost — but you have to see the full version first to feel the trade-off.

**The post-training ladder (where SFT sits):**
```
pre-training ......... learn language from the open web (someone else paid for this)
   |
continued pre-training  inject domain text (optional; not in this workshop)
   |
SFT  <-- you are here .. teach a behavior from instruction/response pairs
   |
preference tuning ..... RLHF / DPO — make it prefer good answers over bad (concept in slides)
```

> **Hardware:** runs on a free Colab **T4**. Use a tiny `max_steps` so it finishes in class. The exact same code with `max_steps=-1` and the full dataset is a real (multi-hour) training run.

## 0. Setup & GPU check

In [ ]:
# Same pinned stack as the LoRA/QLoRA notebooks (02 & 03) for version consistency.
# This notebook trains with plain Transformers + TRL (it does NOT import Unsloth) — see section 1
# for why. ~3-4 min on first run.
%pip install -q unsloth
%pip install -q --no-deps "trl<0.24" peft accelerate bitsandbytes datasets

In [ ]:
import torch

assert torch.cuda.is_available(), (
    "No GPU detected. In Colab: Runtime -> Change runtime type -> T4 GPU, then re-run."
)
gpu = torch.cuda.get_device_properties(0)
print(f"GPU            : {gpu.name}")
print(f"Total VRAM     : {gpu.total_memory / 1024**3:.1f} GB")
print(f"Compute Cap.   : {gpu.major}.{gpu.minor}  "
      f"({'Ampere+ (Flash-Attention 2 path)' if gpu.major >= 8 else 'pre-Ampere (T4): works, slower attention path'})")
print(f"bf16 supported : {torch.cuda.is_bf16_supported()}")

In [ ]:
import os
# Quiet, reproducible runs. (We are NOT using comet/wandb in the workshop.)
os.environ["WANDB_DISABLED"] = "true"

---
## 1. Load the base model — full precision

For full fine-tuning we pass `full_finetuning=True` and do **not** load in 4-bit. We want the real bf16/fp16 weights because we are about to compute a gradient for *every* parameter in the network, not just adapters.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

MAX_SEQ_LEN = 1024            # MedQuAD answers are short-ish; 1024 keeps activations small on T4
MODEL_NAME  = "Qwen/Qwen3-0.6B"

# Full fine-tuning of a small model runs cleanly in plain Transformers + TRL. We deliberately do
# NOT use Unsloth for the FULL fine-tune: its full-finetuning path requires bf16 mixed precision,
# which a free Colab T4 (pre-Ampere) does not support. Plain Transformers trains in fp32 on a T4
# and bf16 on Ampere+ GPUs. (Unsloth still powers the LoRA / QLoRA notebooks, where its memory
# savings matter and the model loads in fp16.)
# NOTE: Transformers 5.x defaults from_pretrained to the checkpoint's native dtype (bf16 for
# Qwen3) — we force float32 so the T4 can train it.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32)
model.gradient_checkpointing_enable()     # trade a little compute for memory — keeps full FT inside 15 GB

n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {n_total/1e6:.1f}M")
print(f"Trainable params: {n_train/1e6:.1f}M  ({100*n_train/n_total:.1f}%)  <-- everything")

---
## 2. Load the prepared dataset

We use the JSONL from Notebook 00. If you ran this notebook fresh (no local file), it pulls a small slice straight from MedQuAD and formats it inline so the cell still runs — but the *real* training set is the structured one you built in NB00.

In [ ]:
import json, os
from pathlib import Path
from datasets import load_dataset, Dataset

SYSTEM_PROMPT = (
    "You are a clinical assistant. Read the medical question and respond with a JSON object "
    "containing: disease, patient_info, symptoms (list), treatment (list), answer_summary."
)

def load_prepared(split):
    p = Path(f"data/medquad_{split}.jsonl")
    if p.exists():
        return Dataset.from_list([json.loads(l) for l in p.read_text().splitlines() if l.strip()])
    return None

train_ds = load_prepared("train")
eval_ds  = load_prepared("validation")

if train_ds is None:
    print("No prepared JSONL found — building a tiny fallback from raw MedQuAD (answers as prose).")
    raw = load_dataset("lavita/MedQuAD", split="train").filter(lambda r: r.get("answer"))
    raw = raw.shuffle(seed=3407).select(range(200))
    def fallback(r):
        return {"messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": r["question"]},
            {"role": "assistant", "content": r["answer"][:800]},
        ]}
    rows = [fallback(r) for r in raw]
    train_ds = Dataset.from_list(rows[:160]); eval_ds = Dataset.from_list(rows[160:])

print(f"train: {len(train_ds)} | eval: {len(eval_ds)}")
print(train_ds[0]["messages"][1])  # a sample user turn

In [ ]:
# Render each conversation through the chat template -> a single 'text' field for the trainer.
def to_text(row):
    return {"text": tokenizer.apply_chat_template(
        row["messages"], tokenize=False, add_generation_prompt=False)}

train_ds = train_ds.map(to_text)
eval_ds  = eval_ds.map(to_text)
print(train_ds[0]["text"][:600])

---
## 3. Configure the trainer

Key SFT choices, and *why*:

- **No packing.** Each example is one self-contained conversation. We want the model to learn the user->assistant boundary independently; packing many conversations into one block corrupts that.
- **Prompt masking is automatic.** `SFTTrainer` masks the system+user tokens so the loss is computed **only on the assistant response** — the model is graded on what it should *generate*, not on echoing the prompt.
- **Low learning rate (2e-5).** Touching all the weights risks *catastrophic forgetting* of general ability. A small LR keeps the update gentle. (LoRA in NB02 can afford 10x more because it only nudges small adapters.)

In [ ]:
from trl import SFTTrainer, SFTConfig
import torch

# Precision that works everywhere: bf16 on Ampere+ (A100/L4), pure fp32 on a T4.
# (No fp16 AMP — full fine-tuning in fp16 risks gradient overflow.)
use_bf16 = torch.cuda.is_bf16_supported()

cfg = SFTConfig(
    output_dir="outputs_full_sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,        # effective batch = 8
    warmup_steps=5,
    max_steps=60,                          # DEMO: finishes in class. Set -1 + num_train_epochs=1 for real.
    learning_rate=2e-5,                    # gentle — full FT forgets if you go big
    fp16=False,
    bf16=use_bf16,
    optim="adamw_8bit",                    # 8-bit optimizer states -> less VRAM
    weight_decay=0.01,
    logging_steps=2,
    eval_strategy="steps",
    eval_steps=30,
    save_strategy="no",
    seed=3407,
    report_to="none",
    max_length=MAX_SEQ_LEN,                # TRL arg is max_length (Unsloth's patched config used max_seq_length)
    dataset_text_field="text",
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,            # TRL arg is processing_class (older/Unsloth builds used tokenizer)
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=cfg,
)

---
## 4. Train — and watch the four VRAM "buckets"

When `trainer.train()` runs, GPU memory fills in four buckets:

| Bucket | What | Depends on |
|---|---|---|
| 1. Weights | the 0.6B parameters | model size |
| 2. Gradients | one per trainable weight | **trainable** params (= all, here) |
| 3. Optimizer states | Adam's momentum + variance | trainable params |
| 4. Activations | the forward-pass math | batch size x sequence length |

Buckets 2 & 3 are why **full** fine-tuning is heavy: you pay for a gradient *and* two optimizer states for every single weight. LoRA (NB02) collapses buckets 2 & 3 to almost nothing. Watch the peak-VRAM number — we'll compare it across all three methods in NB03.

In [ ]:
import torch
torch.cuda.reset_peak_memory_stats()
start = torch.cuda.memory_allocated() / 1024**3
print(f"Pre-train VRAM (weights+optim loaded): {start:.2f} GB")

stats = trainer.train()

peak = torch.cuda.max_memory_allocated() / 1024**3
print(f"\nFinal train loss : {stats.training_loss:.4f}")
print(f"Peak VRAM        : {peak:.2f} GB  <-- remember this for the NB03 comparison")

# Stash the result so NB03's comparison table can read it back.
import json
from pathlib import Path
Path("data").mkdir(exist_ok=True)
Path("data/run_full_sft.json").write_text(json.dumps(
    {"method": "full_sft", "trainable_pct": 100.0, "peak_vram_gb": round(peak, 2),
     "final_loss": round(float(stats.training_loss), 4), "steps": stats.global_step}, indent=2))

---
## 5. Does it work? Before -> after

A loss going down is necessary but not sufficient. Let's actually ask the model a held-out medical question and look at the output. (Rigorous, metric-based evaluation is all of Week 5 — this is just the eyeball check.)

In [ ]:
from transformers import AutoModelForCausalLM, TextStreamer

def ask(m, question, max_new_tokens=200):
    m.config.use_cache = True                          # KV cache on for generation (training turned it off)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt", return_dict=True).to(m.device)   # Transformers 5.x: returns a BatchEncoding
    out = m.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.3)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

Q = "What are the symptoms and treatment of Hashimoto's disease?"

# BEFORE — a fresh copy of the ORIGINAL weights (no fine-tuning at all)
base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(model.device)
print("=== BASE (before fine-tuning) ===")
print(ask(base, Q))
del base; torch.cuda.empty_cache()                     # free it before generating from the tuned model

# AFTER — `model`, whose 100% of weights you just updated in section 4
print("\n=== FINE-TUNED (after full SFT) ===")
print(ask(model, Q))

---
## 6. Save the model

Full fine-tuning produces a **standalone** model (all weights), ~1.2 GB for 0.6B. We save locally; pushing to the Hub is optional (uncomment).

In [ ]:
model.save_pretrained("qwen3-medquad-full-sft")
tokenizer.save_pretrained("qwen3-medquad-full-sft")
print("Saved to ./qwen3-medquad-full-sft")

# Optional Hub push (needs HF_TOKEN in Colab Secrets):
# model.push_to_hub("YOUR_USERNAME/qwen3-medquad-full-sft", token=os.environ["HF_TOKEN"])
# tokenizer.push_to_hub("YOUR_USERNAME/qwen3-medquad-full-sft", token=os.environ["HF_TOKEN"])

## What you learned

You updated every weight of a real language model and changed its behavior. You also felt the cost: full gradients + optimizer states for all params, the heaviest VRAM footprint of the three methods.

**Next — `02_lora.ipynb`:** freeze the base model, train tiny adapter matrices instead, and reach comparable quality while training **under 1%** of the parameters.